In [0]:
%pip install pandas

In [0]:
df = spark.table("`data-growth-analytic`.bronze.rascunho")
display(df)

In [0]:
print(df)

In [0]:
#Convertendo as colunas para o tipo correto
df = df.withColumn("discount_pct", df["discount_pct"].cast("float"))
df = df.withColumn("quantity", df["quantity"].cast("int"))
df = df.withColumn("unit_price", df["unit_price"].cast("float"))
df = df.withColumn("sent", df["sent"].cast("int"))
df = df.withColumn("opened", df["opened"].cast("int"))
df = df.withColumn("clicked", df["clicked"].cast("int"))
df = df.withColumn("converted", df["converted"].cast("int"))
df = df.withColumn("sent_date", df["sent_date"].cast("date"))
df = df.withColumn("order_date", df["order_date"].cast("date"))
df = df.withColumn("signup_date", df["signup_date"].cast("date"))
#Renomeando as colunas
df = df.withColumnRenamed("age_band", "idade")
df = df.withColumnRenamed("discount_pct","desconto_pct")
df = df.withColumnRenamed("quantity","quantidade")
df = df.withColumnRenamed("unit_price","preco_unitario")
df = df.withColumnRenamed("sent","enviado")
df = df.withColumnRenamed("opened","aberto")
df = df.withColumnRenamed("clicked","clicado")
df = df.withColumnRenamed("converted","convertido")
df = df.withColumnRenamed("sent_date","data_envio")
df = df.withColumnRenamed("order_date","data_pedido")
df = df.withColumnRenamed("signup_date","data_cadastro")
df = df.withColumnRenamed("customer_id", "id_cliente")
df = df.withColumnRenamed("acquisition_channel", "canal_acesso")
df = df.withColumnRenamed("region", "regiao")
df = df.withColumnRenamed("customer_segment", "segmento_cliente")
df = df.withColumnRenamed("order_id", "id_pedido")
df = df.withColumnRenamed("product_id", "id_produto")
df = df.withColumnRenamed("product_category", "categoria_produto")
df = df.withColumnRenamed("product_name", "nome_produto")
df = df.withColumnRenamed("order_status", "status_pedido")
df = df.withColumnRenamed("payment_method", "metodo_pagamento")
df = df.withColumnRenamed("order_purchase_timestamp", "data_compra")
df = df.withColumnRenamed("order_delivered_customer_date", "data_entrega")
df = df.withColumnRenamed("order_estimated_delivery_date", "data_estimada_entrega")
df = df.withColumnRenamed("campaign_name", "nome_campanha")
df = df.withColumnRenamed("campaign_channel", "canal_campanha")
df = df.withColumnRenamed("gross_item_revenue", "revenue")

In [0]:
df.printSchema()

In [0]:
display(df[['id_cliente','id_pedido','id_produto']])

In [0]:
from pyspark.sql.functions import col

colunas = ['id_cliente', 'id_pedido', 'id_produto']

for coluna in colunas:
    df = df.withColumn(coluna, col(coluna).cast("int"))

In [0]:
df.printSchema()#

In [0]:
#Identificando valores nulos
from pyspark.sql.functions import col, sum, when

df.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df.columns
]).show()

In [0]:
df = df.withColumnRenamed("revenue", "receita")

In [0]:
df.select("quantidade","preco_unitario","receita","desconto_pct","enviado","aberto","clicado","convertido").summary().show()

In [0]:
total_linhas = df.count()
total_distintas = df.distinct().count()

if total_linhas > total_distintas:
    print(f"Existem {total_linhas - total_distintas} linhas duplicadas!")
else:
    print("Não há linhas duplicadas.")

In [0]:
# Mostra as linhas idênticas que aparecem mais de uma vez
df.groupBy(df.columns).count().filter("count > 1").show()


In [0]:
# 1. Descobre quais IDs estão duplicados
ids_duplicados = df.groupBy("id_cliente").count().filter("count > 1")

# 2. Faz um join de volta no DF original para ver todas as linhas desses IDs
df.join(ids_duplicados, on="id_cliente", how="inner").show()


In [0]:
df_limpo = df.dropDuplicates(["id_cliente"]) 


In [0]:
total_linhas = df_limpo.count()
total_distintas = df_limpo.distinct().count()

if total_linhas > total_distintas:
    print(f"Existem {total_linhas - total_distintas} linhas duplicadas!")
else:
    print("Não há linhas duplicadas.")

In [0]:
# Criar o data frame df_limpo no volume silver
df_limpo.write.mode("overwrite").saveAsTable("`data-growth-analytic`.silver.df_limpo")